# Módulo 44: Preparación de datos — Solución
*Juliho Castillo*

Enunciado para estudiantes: [m44-preparacion-de-datos-guia.ipynb](m44-preparacion-de-datos-guia.ipynb) · Lección: [m44-preparacion-de-datos-leccion.ipynb](m44-preparacion-de-datos-leccion.ipynb)

## Contexto

Una cadena de tiendas en diez países entrega un extracto de ventas (CSV de 2019, una carpeta con los CSV de 2020 y 2021, un TXT de países y un Excel con clientes, tiendas y productos). Hay que limpiarlo con pandas hasta tener una tabla de hechos y cuatro dimensiones con claves únicas, listas para el modelo del Módulo 45. El enunciado original del curso se perdió; las instrucciones de la guía se reconstruyeron a partir de la solución.

In [1]:
from pathlib import Path

import pandas as pd

DATOS = Path("../practica-ventas/datos-ventas")

### Tarea 1. Países

In [2]:
paises = (
    pd.read_csv(DATOS / "paises.txt", sep="|", header=None, names=["id_pais", "pais"])
    .drop_duplicates()
    .sort_values("id_pais", ignore_index=True)
)
print(len(paises), "países; clave única:", paises["id_pais"].is_unique)
paises

10 países; clave única: True


,id_pais,pais
0,1,US
1,2,CANADA
2,3,MEXICO
3,4,BRAZIL
4,5,SPAIN
5,6,COLOMBIA
6,7,CHILE
7,8,PERU
8,9,PORTUGAL
9,10,FRANCE


El archivo tiene 13 filas porque `US`, `BRAZIL` y `SPAIN` aparecen dos veces; tras `drop_duplicates` quedan 10 países.

### Tarea 2. Clientes

In [3]:
crudo = pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_customer", dtype=str)
# Las filas válidas empiezan por "ID|": así se van las vacías y el pie del extracto
validas = crudo[crudo["ID_Client"].str.startswith("ID|", na=False)]
partes = validas["ID_Client"].str.split("|", expand=True)
nombre = partes[2].str.split("/", n=1, expand=True)

clientes = pd.DataFrame({
    "id_cliente": partes[1].astype(int),
    "nombre": nombre[0],
    "apellido": nombre[1],
    "correo": partes[3].str.removeprefix("mail:"),
    "telefono": partes[4],
}).sort_values("id_cliente", ignore_index=True)

print(len(crudo), "filas en la hoja ->", len(clientes), "clientes; clave única:", clientes["id_cliente"].is_unique)
clientes.head()

955 filas en la hoja -> 950 clientes; clave única: True


,id_cliente,nombre,apellido,correo,telefono
0,1,Melany,Morrad,mmorrad0@walmart.com,+63-253-635-2782
1,2,Cally,Maginot,cmaginot1@google.ca,+86-574-743-6664
2,3,Elwyn,Timmes,etimmes2@google.de,+46-846-858-9359
3,4,Marlie,Edmott,medmott3@apache.org,+62-716-339-1045
4,5,Jillane,Beaushaw,jbeaushaw4@vk.com,+63-715-207-9524


La columna `zone` se descarta porque vale `?` en todas las filas, y `first_name`/`last_name` repiten lo que ya trae el campo empaquetado.

### Tarea 3. Productos

In [4]:
crudo = pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_product")
crudo["id_producto"] = crudo["Product ID"].str.extract(r"ID-(\d+)\|", expand=False).astype(int)
print("Claves repetidas antes de filtrar:", crudo["id_producto"].duplicated().sum())

productos = (
    crudo[crudo["Active"] == 1]
    .rename(columns={"Category": "categoria", "Sub-Category": "subcategoria", "Product Name": "producto"})
    [["id_producto", "categoria", "subcategoria", "producto"]]
    .sort_values("id_producto", ignore_index=True)
)
print(len(productos), "productos activos; clave única:", productos["id_producto"].is_unique)
productos.head()

Claves repetidas antes de filtrar: 17
1000 productos activos; clave única: True


,id_producto,categoria,subcategoria,producto
0,1,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...
1,2,Furniture,Chairs,"Novimex Executive Leather Armchair, Black"
2,3,Technology,Phones,"Nokia Smart Phone, with Caller ID"
3,4,Technology,Phones,"Motorola Smart Phone, Cordless"
4,5,Technology,Copiers,"Sharp Wireless Fax, High-Speed"


### Tarea 4. Tiendas

In [5]:
crudo = pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_store", header=2)
print("Formatos originales:", crudo["Format"].map(repr).unique().tolist())

abiertas = crudo[crudo["Status"] == "Open"]
tiendas = pd.DataFrame({
    "id_tienda": abiertas["ID_Store"].astype(int),
    "tienda": abiertas["Name_Store"],
    "direccion": abiertas["Addres"],
    "formato": abiertas["Format"].str.strip().str.upper(),
}).sort_values("id_tienda", ignore_index=True)

print(len(crudo), "filas ->", len(tiendas), "tiendas; clave única:", tiendas["id_tienda"].is_unique)
print("Formatos limpios:", tiendas["formato"].value_counts().to_dict())

Formatos originales: ["'AAA'", "'aa'", "'   A'"]


204 filas -> 200 tiendas; clave única: True
Formatos limpios: {'AA': 71, 'AAA': 69, 'A': 60}


Las tiendas 17, 167, 188 y 197 aparecían como `Open` y como `Closed`; al quedarnos con `Open` queda una fila por tienda.

### Tarea 5. Ventas de 2019

In [6]:
def leer_ventas_csv(ruta):
    ventas = pd.read_csv(ruta, encoding="utf-8-sig", usecols=range(8))
    ventas["Date"] = pd.to_datetime(ventas["Date"], format="%m/%d/%Y")
    return ventas.drop(columns="ID_T")


ventas_2019 = leer_ventas_csv(DATOS / "ventas-2019.csv")
print(ventas_2019.shape, ventas_2019["Date"].min().date(), ventas_2019["Date"].max().date())
ventas_2019.dtypes

(300000, 7) 2019-01-01 2019-12-31


ID_Country             int64
Date          datetime64[ns]
ID_Store               int64
ID_Client              int64
ID_Product             int64
Quantity               int64
Price                  int64
dtype: object

### Tarea 6. Ventas de 2020 y 2021 desde una carpeta

In [7]:
archivos = sorted((DATOS / "ventas-por-anio").glob("*.csv"))
ventas_2020_2021 = pd.concat([leer_ventas_csv(archivo) for archivo in archivos], ignore_index=True)
print([archivo.name for archivo in archivos])
print(ventas_2020_2021.shape, "columnas:", ventas_2020_2021.columns.tolist())

['ventas-2020.csv', 'ventas-2021.csv']
(1199000, 7) columnas: ['ID_Country', 'Date', 'ID_Store', 'ID_Client', 'ID_Product', 'Quantity', 'Price']


`usecols=range(8)` evita la columna vacía que produce la coma final de `ventas-2021.csv`.

### Tarea 7. Tabla de ventas única

In [8]:
ventas = pd.concat([ventas_2019, ventas_2020_2021], ignore_index=True).rename(columns={
    "Date": "fecha", "ID_Country": "id_pais", "ID_Store": "id_tienda", "ID_Client": "id_cliente",
    "ID_Product": "id_producto", "Quantity": "cantidad", "Price": "precio",
})[["fecha", "id_pais", "id_tienda", "id_cliente", "id_producto", "cantidad", "precio"]]
ventas["importe"] = ventas["cantidad"] * ventas["precio"]

print(f"{len(ventas):,} ventas del {ventas['fecha'].min():%Y-%m-%d} al {ventas['fecha'].max():%Y-%m-%d}".replace(",", " "))
print(f"Ventas totales: {ventas['importe'].sum():,}".replace(",", " "))
ventas.head()

1 499 000 ventas del 2019-01-01 al 2021-12-31
Ventas totales: 41 940 958 825


,fecha,id_pais,id_tienda,id_cliente,id_producto,cantidad,precio,importe
0,2019-10-24,9,1,551,291,155,233,36115
1,2019-06-17,2,153,774,710,25,415,10375
2,2019-03-04,10,159,356,962,445,58,25810
3,2019-05-15,1,74,500,768,8,179,1432
4,2019-07-14,6,61,1,649,36,85,3060


In [9]:
import sys

sys.path.insert(0, str(Path("../practica-ventas").resolve()))
import ventas as practica

for mia, referencia in [
    (ventas, practica.leer_ventas()),
    (paises, practica.leer_paises()),
    (clientes, practica.leer_clientes()),
    (productos, practica.leer_productos()),
    (tiendas, practica.leer_tiendas()),
]:
    pd.testing.assert_frame_equal(mia, referencia)
print("Las cinco tablas coinciden con ventas.py.")

Las cinco tablas coinciden con ventas.py.


Las cinco tablas tienen claves únicas y enteras, las ventas cubren del 1 de enero de 2019 al 31 de diciembre de 2021 (1 499 000 filas) y suman 41 940 958 825. Son las mismas que produce `ventas.py`, de donde leen los módulos siguientes.

**Revisión posterior (2026-10):** El ejercicio se resolvió originalmente en Power BI (Power Query), con capturas de pantalla como evidencia. Se rehízo en Python con pandas para que la sección no dependa de software de pago ni de Windows; la versión original sigue en el historial de git.